# Task 3: Build & Evaluate a Machine Learning Model
## AI/ML Internship — Week 3

**Project:** Student Pass/Fail Prediction

**Objective:** Train and compare two classification models to predict whether a student passes based on available student and study-related information.

> **Important:** This notebook uses the UCI Student Performance dataset as a starter example. If your Week 2 task used a different dataset, replace the dataset-loading section and update the target/features so this uses the same cleaned dataset as your Week 2 submission.

### Steps covered
1. Define the problem statement.
2. Load and inspect the dataset.
3. Select input features (X) and target (y).
4. Split data into training and testing sets.
5. Train Logistic Regression and Random Forest models.
6. Generate predictions.
7. Evaluate using accuracy, precision, recall, F1-score, and confusion matrix.
8. Compare models and explain the results.


## 1. Problem Statement

Student performance depends on several factors, such as study time, absences, family support, and previous grades. The aim is to build a classification model that predicts whether a student passes the course.

**Target definition:** A student is classified as **Pass** when the final grade (`G3`) is 10 or higher (on the dataset's 0–20 scale); otherwise, the student is classified as **Fail**.

**Note:** This is an educational demonstration, not a system for making real decisions about students.


In [ ]:
# Import required libraries
import io
import zipfile
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

print("Libraries imported successfully.")


## 2. Load the Dataset

The code below downloads the public UCI Student Performance dataset. If you have your cleaned Week 2 CSV, you can instead upload it to Colab and replace the download code with:

```python
df = pd.read_csv('/content/your_cleaned_dataset.csv')
```


In [ ]:
# Download the UCI Student Performance dataset
url = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"

try:
    with urllib.request.urlopen(url, timeout=30) as response:
        zip_bytes = response.read()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        # Student Mathematics dataset
        with z.open("student-mat.csv") as f:
            df = pd.read_csv(f, sep=";")
    print("Dataset downloaded successfully.")
except Exception as e:
    print("Automatic download did not work:", e)
    print("In Google Colab, upload your Week 2 cleaned CSV and run:")
    print("df = pd.read_csv('/content/your_cleaned_dataset.csv')")
    raise

print("Dataset shape (rows, columns):", df.shape)
display(df.head())


In [ ]:
# Inspect the dataset
print("Column names:")
print(df.columns.tolist())

print("\\nDataset information:")
df.info()

print("\\nMissing values per column:")
display(df.isnull().sum().to_frame("Missing values"))

print("\\nDuplicate rows:", df.duplicated().sum())


## 3. Prepare the Data and Define X and y

- **X (features):** student information and study-related variables.
- **y (target):** Pass/Fail label derived from final grade `G3`.

We remove `G1`, `G2`, and `G3` from the input features. This avoids using earlier/final grade columns that could make the prediction unrealistically easy and keeps the model focused on other student factors.


In [ ]:
# Basic cleaning
df = df.drop_duplicates().copy()
df = df.dropna(subset=["G3"]).copy()

# Create the target: 1 = Pass, 0 = Fail
df["pass"] = (df["G3"] >= 10).astype(int)

# Avoid grade columns as inputs to prevent target leakage
columns_to_drop = ["G1", "G2", "G3", "pass"]
X = df.drop(columns=columns_to_drop)
y = df["pass"]

print("Feature matrix X shape:", X.shape)
print("Target y shape:", y.shape)
print("\\nTarget distribution (0 = Fail, 1 = Pass):")
display(y.value_counts().rename(index={0: "Fail", 1: "Pass"}).to_frame("Students"))


## 4. Split into Training and Testing Sets

We use 80% of the data for training and 20% for testing. `stratify=y` helps keep the Pass/Fail proportions similar in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 5. Preprocess and Train Two Models

Some columns contain text categories, while others contain numbers. We handle both automatically:
- Missing numeric values are filled with the median and numeric values are scaled.
- Missing categorical values are filled with the most common value and categories are one-hot encoded.

We train:
1. **Logistic Regression** — a common baseline classification algorithm.
2. **Random Forest Classifier** — combines multiple decision trees to make predictions.


In [ ]:
# Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
}

trained_models = {}
predictions = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline
    predictions[name] = pipeline.predict(X_test)
    print(name, "trained successfully.")


## 6. Generate Predictions

The models predict Pass (1) or Fail (0) for the held-out test records.


In [ ]:
for name, y_pred in predictions.items():
    print(f"\\n{name} — first 10 predictions:")
    print(y_pred[:10])

# Show actual vs predicted values for the first 10 test records
comparison = pd.DataFrame({
    "Actual": y_test.iloc[:10].map({0: "Fail", 1: "Pass"}).values,
    "Logistic Regression": pd.Series(predictions["Logistic Regression"][:10]).map({0: "Fail", 1: "Pass"}).values,
    "Random Forest": pd.Series(predictions["Random Forest"][:10]).map({0: "Fail", 1: "Pass"}).values
})
display(comparison)


## 7. Evaluate the Models

We use:
- **Accuracy:** proportion of all predictions that are correct.
- **Precision:** among predicted passes, how many were actually passes.
- **Recall:** among actual passes, how many were correctly identified.
- **F1-score:** balance between precision and recall.
- **Confusion matrix:** counts of correct and incorrect predictions.


In [ ]:
results = []

for name, y_pred in predictions.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0)
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(3))

for name, y_pred in predictions.items():
    print(f"\\nClassification report — {name}")
    print(classification_report(y_test, y_pred, target_names=["Fail", "Pass"], zero_division=0))


In [ ]:
# Plot confusion matrices
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fail", "Pass"])
    disp.plot()
    plt.title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()


## 8. Compare the Models

The table below compares both models using the evaluation metrics. The model with the highest F1-score is selected as the best model for this run. Review the confusion matrices and precision/recall as well, rather than relying on accuracy alone.


In [ ]:
best_model_name = results_df["F1-score"].idxmax()
print("Best model based on F1-score:", best_model_name)
print("\\nComparison table:")
display(results_df.sort_values("F1-score", ascending=False).round(3))


## 9. Results and Analysis

After running all cells, fill in the actual metric values from the comparison table.

**Report-ready explanation:**

- Two classification algorithms were implemented: Logistic Regression and Random Forest.
- The dataset was divided into 80% training data and 20% testing data.
- Both models generated Pass/Fail predictions for the test set.
- Accuracy, precision, recall, F1-score, and confusion matrices were used to evaluate performance.
- The model with the higher F1-score was selected as the best-performing model in this experiment.
- Performance may vary depending on the train-test split and dataset quality. The model should not be used to make real-world decisions about individual students.

**Write your final result after running the notebook:**  
“Logistic Regression achieved an accuracy of ___ and an F1-score of ___. Random Forest achieved an accuracy of ___ and an F1-score of ___. Based on the F1-score, ___ performed better on this test split.”


## 10. Conclusion

This project demonstrated a basic Machine Learning workflow: loading data, cleaning it, selecting features and a target, splitting the data, training two models, generating predictions, and evaluating the results. The comparison shows how different algorithms can perform differently on the same dataset.

### Future Improvements
- Use the exact cleaned dataset from Week 2.
- Try cross-validation and tune model parameters.
- Collect more representative data and evaluate possible bias.
- Document dataset limitations and avoid using predictions as the sole basis for decisions about students.

### Submission Checklist
- [ ] Jupyter Notebook (`.ipynb`)
- [ ] Dataset (`.csv`) and its source
- [ ] Trained models or saved pipeline (optional but useful)
- [ ] Evaluation results and model comparison
- [ ] Short README file
- [ ] Screenshots of output and graphs
